# Track A notebook: CFD7 → OpenFOAM → Python GIF

Run this notebook inside **WSL Ubuntu 22.04**. Execute the large setup cell first, then run `main()` in the last cell.

This v1 solves **airflow U/p** with OpenFOAM, converts output with `foamToVTK`, then exports velocity and particle GIFs with PyVista. Temperature and moving body are intentionally left for v2.


In [ ]:
"""
Track A: CFD7 -> OpenFOAM solver -> Python/PyVista GIF

Run this script inside WSL Ubuntu, not normal Windows Python.

Recommended:
    wsl -d Ubuntu-22.04
    source /opt/openfoam13/etc/bashrc
    pip install pyvista tqdm imageio
    python CFD7_trackA_openfoam_python_animation.py

This v1 solves airflow U/p with OpenFOAM, converts result with foamToVTK,
then creates velocity and particle GIFs with Python/PyVista.
"""

import os
import re
import shlex
import shutil
import subprocess
import platform
from pathlib import Path

import numpy as np
import pyvista as pv
from tqdm.auto import tqdm

try:
    pv.OFF_SCREEN = True
except Exception:
    pass

# ============================================================
# 0. Safety check
# ============================================================
if platform.system().lower().startswith("win"):
    raise RuntimeError(
        "Run this script inside WSL Ubuntu, not normal Windows Python. "
        "OpenFOAM is installed inside WSL."
    )

# ============================================================
# 1. CFD7 parameters copied/adapted from your original notebook
# ============================================================

Lx = 4.0
Ly = 4.0
Lz = 4.0

# Original CFD7 grid was 41 x 41 x 41 points -> 40 cells per direction.
nx = 41
ny = 41
nz = 41

dx = Lx / (nx - 1)
dy = Ly / (ny - 1)
dz = Lz / (nz - 1)

rho = 1.0       # kept for particle model
nu = 0.03       # m^2/s, same as CFD7
U_in = 2.0      # m/s, inlet points downward in -z direction


def windows_path_to_wsl(path_string: str) -> Path:
    """Convert C:\\Users\\... to /mnt/c/Users/... when running in WSL."""
    s = str(path_string)
    m = re.match(r"^([A-Za-z]):\\(.*)$", s)
    if not m:
        return Path(s)
    drive = m.group(1).lower()
    rest = m.group(2).replace("\\", "/")
    return Path(f"/mnt/{drive}/{rest}")


# Change these if your files are somewhere else.
cad_file_path = windows_path_to_wsl(
    r"C:\Users\brian\OneDrive\桌面\ASE\cleanroom\simulation obstacles\tinker.obj"
)

save_folder = windows_path_to_wsl(
    r"C:\Users\brian\OneDrive\桌面\ASE\cleanroom\animation"
)
save_folder.mkdir(parents=True, exist_ok=True)

# Your note: raw OBJ longest side about 152 units, scale to ~4 m.
cad_unit_scale = 4.0 / 152.0
cad_random_seed = 7
cad_obstacle_mode = "shell"

# Inlet geometry from CFD7
inlet_diameter = 0.81
inlet_radius = inlet_diameter / 2.0
inlet_area = np.pi * inlet_radius**2
inlet_center_x = Lx / 2.0
inlet_center_y = Ly / 2.0

# blockMesh cannot make a true circular patch easily.
# v1 uses a square patch centered at the same place.
inlet_patch_side = inlet_diameter
inlet_x_min = inlet_center_x - inlet_patch_side / 2.0
inlet_x_max = inlet_center_x + inlet_patch_side / 2.0
inlet_y_min = inlet_center_y - inlet_patch_side / 2.0
inlet_y_max = inlet_center_y + inlet_patch_side / 2.0

# Outlet geometry from CFD7
outlet_floor_fraction = 0.17
outlet_strip_length_x = Lx
outlet_area = inlet_area * outlet_floor_fraction
outlet_strip_width_y = outlet_area / outlet_strip_length_x
outlet_x_min = 0.0
outlet_x_max = Lx
outlet_center_y = Ly / 2.0
outlet_y_min = outlet_center_y - outlet_strip_width_y / 2.0
outlet_y_max = outlet_center_y + outlet_strip_width_y / 2.0

# CFD7 outlet strip is ~2 cm wide, smaller than the original 0.1 m grid spacing.
# v1 uses a minimum 0.10 m meshed outlet patch to avoid very bad cells.
MIN_OUTLET_WIDTH_FOR_MESH = 0.10
outlet_y_min_mesh = outlet_center_y - max(outlet_strip_width_y, MIN_OUTLET_WIDTH_FOR_MESH) / 2.0
outlet_y_max_mesh = outlet_center_y + max(outlet_strip_width_y, MIN_OUTLET_WIDTH_FOR_MESH) / 2.0

# Particle parameters copied from CFD7
max_particles = 50
release_every = 5
particles_per_release = 1
particle_release_z = Lz - 0.05
particle_density = 1000.0
air_dynamic_viscosity = 1.8e-5
g_particle = np.array([0.0, 0.0, -9.81])
particle_diameter_options = np.array([0.3e-6, 0.5e-6, 1.0e-6])
particle_diameter_probabilities = np.array([0.70, 0.24, 0.06])

# OpenFOAM setup
OF_VERSION = "13"
FOAM_BASHRC = f"/opt/openfoam{OF_VERSION}/etc/bashrc"
user_name = os.environ.get("USER", "brian")
case_dir = Path.home() / "OpenFOAM" / f"{user_name}-{OF_VERSION}" / "run" / "cleanroomTrackA_CFD7"

# Turn on only after blockMesh-only works and your STL is clean/watertight.
USE_SNAPPY_CAD_OBSTACLE = False

# Steady OpenFOAM iterations. This is pseudo-time for a steady solver.
of_end_time = 300
of_write_interval = 25

# ============================================================
# 2. CAD helper functions adapted from CFD7
# ============================================================


def load_and_prepare_cad_mesh(cad_path: Path, unit_scale: float) -> pv.PolyData:
    """Load CAD mesh, scale it, extract surface, triangulate, and clean."""
    if not cad_path.exists():
        raise FileNotFoundError(
            f"CAD file not found:\n{cad_path}\n\n"
            "Check cad_file_path. If running in WSL, Windows C: paths must become /mnt/c/..."
        )

    mesh = pv.read(str(cad_path))
    if isinstance(mesh, pv.MultiBlock):
        mesh = mesh.combine()

    mesh = mesh.extract_surface().triangulate().clean()

    if unit_scale != 1.0:
        mesh.points *= unit_scale

    return mesh


def choose_random_4m_cube_from_cad(mesh: pv.PolyData, seed: int):
    """Pick a 4 m x 4 m x 4 m region from the CAD, same idea as CFD7."""
    rng_local = np.random.default_rng(seed)
    pts = mesh.points

    if pts.shape[0] == 0:
        raise RuntimeError("CAD mesh has no points.")

    random_point = pts[rng_local.integers(0, pts.shape[0])]

    target_length = np.array([Lx, Ly, Lz])
    cube_min = random_point - target_length / 2.0
    cube_max = cube_min + target_length

    xmin, xmax, ymin, ymax, zmin, zmax = mesh.bounds
    cad_min = np.array([xmin, ymin, zmin])
    cad_max = np.array([xmax, ymax, zmax])
    cad_length = cad_max - cad_min

    for d in range(3):
        if cad_length[d] >= target_length[d]:
            if cube_min[d] < cad_min[d]:
                cube_min[d] = cad_min[d]
                cube_max[d] = cube_min[d] + target_length[d]
            if cube_max[d] > cad_max[d]:
                cube_max[d] = cad_max[d]
                cube_min[d] = cube_max[d] - target_length[d]

    return (cube_min[0], cube_max[0], cube_min[1], cube_max[1], cube_min[2], cube_max[2])


def crop_cad_to_selected_bounds(mesh: pv.PolyData, selected_bounds):
    """Crop CAD to the selected 4 m box."""
    xmin, xmax, ymin, ymax, zmin, zmax = selected_bounds
    pts = mesh.points

    point_inside = (
        (pts[:, 0] >= xmin) & (pts[:, 0] <= xmax) &
        (pts[:, 1] >= ymin) & (pts[:, 1] <= ymax) &
        (pts[:, 2] >= zmin) & (pts[:, 2] <= zmax)
    )

    if not np.any(point_inside):
        raise RuntimeError(
            "The selected CAD region contains no CAD points. "
            "Try changing cad_random_seed or cad_unit_scale."
        )

    cropped = mesh.extract_points(point_inside, adjacent_cells=True, include_cells=True)
    return cropped.extract_surface().triangulate().clean()


def map_cad_crop_to_simulation_domain(cropped_mesh: pv.PolyData, selected_bounds):
    """Move selected CAD cube into OpenFOAM simulation coordinates [0,4]^3."""
    xmin, xmax, ymin, ymax, zmin, zmax = selected_bounds
    sim_mesh = cropped_mesh.copy()
    sim_mesh.points[:, 0] -= xmin
    sim_mesh.points[:, 1] -= ymin
    sim_mesh.points[:, 2] -= zmin
    return sim_mesh.extract_surface().triangulate().clean()


def prepare_cad_for_case():
    """Load OBJ/STL, crop/map to [0,4]^3, and export STL for OpenFOAM."""
    tri_surface_dir = case_dir / "constant" / "triSurface"
    tri_surface_dir.mkdir(parents=True, exist_ok=True)

    cad_full_mesh = load_and_prepare_cad_mesh(cad_file_path, cad_unit_scale)
    selected_bounds = choose_random_4m_cube_from_cad(cad_full_mesh, cad_random_seed)
    cropped = crop_cad_to_selected_bounds(cad_full_mesh, selected_bounds)
    sim_mesh = map_cad_crop_to_simulation_domain(cropped, selected_bounds)

    stl_path = tri_surface_dir / "cadObstacle.stl"
    sim_mesh.save(str(stl_path), binary=False)

    print("CAD prepared.")
    print("Original scaled CAD bounds:", cad_full_mesh.bounds)
    print("Selected CAD bounds:", selected_bounds)
    print("Mapped CAD bounds:", sim_mesh.bounds)
    print("Wrote STL:", stl_path)

    return sim_mesh, stl_path

# ============================================================
# 3. OpenFOAM dictionary writers
# ============================================================

FOAM_HEADER_TEMPLATE = """FoamFile
{{
    version     2.0;
    format      ascii;
    class       {cls};
    object      {obj};
}}
// ************************************************************************* //
"""


def foam_header(cls: str, obj: str) -> str:
    return FOAM_HEADER_TEMPLATE.format(cls=cls, obj=obj)


def write_text(path: Path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def clean_case_folder() -> None:
    if case_dir.exists():
        shutil.rmtree(case_dir)
    (case_dir / "0").mkdir(parents=True, exist_ok=True)
    (case_dir / "constant").mkdir(parents=True, exist_ok=True)
    (case_dir / "system").mkdir(parents=True, exist_ok=True)


def make_planes():
    # x planes isolate the square inlet. Outlet covers all x.
    x_vals = [0.0, inlet_x_min, inlet_x_max, Lx]

    # y planes isolate both inlet square and outlet strip.
    y_vals = [0.0, inlet_y_min, outlet_y_min_mesh, outlet_y_max_mesh, inlet_y_max, Ly]

    x_planes = []
    for val in sorted(x_vals):
        val = max(0.0, min(val, Lx))
        if not x_planes or abs(val - x_planes[-1]) > 1e-6:
            x_planes.append(val)

    y_planes = []
    for val in sorted(y_vals):
        val = max(0.0, min(val, Ly))
        if not y_planes or abs(val - y_planes[-1]) > 1e-6:
            y_planes.append(val)

    return x_planes, y_planes


def segment_cells(a: float, b: float, target_h: float) -> int:
    return max(1, int(round((b - a) / target_h)))


def write_block_mesh_dict() -> None:
    """Create a multi-block box mesh with separate inlet/outlet patches."""
    x_planes, y_planes = make_planes()
    z_planes = [0.0, Lz]

    verts = []
    vid = {}
    for k, zval in enumerate(z_planes):
        for j, yval in enumerate(y_planes):
            for i, xval in enumerate(x_planes):
                vid[(i, j, k)] = len(verts)
                verts.append((xval, yval, zval))

    def v(i, j, k):
        return vid[(i, j, k)]

    blocks = []
    inlet_faces = []
    outlet_faces = []
    wall_faces = []

    nxseg = len(x_planes) - 1
    nyseg = len(y_planes) - 1

    for j in range(nyseg):
        for i in range(nxseg):
            x0, x1 = x_planes[i], x_planes[i + 1]
            y0, y1 = y_planes[j], y_planes[j + 1]
            xc = 0.5 * (x0 + x1)
            yc = 0.5 * (y0 + y1)

            v0 = v(i,     j,     0)
            v1 = v(i + 1, j,     0)
            v2 = v(i + 1, j + 1, 0)
            v3 = v(i,     j + 1, 0)
            v4 = v(i,     j,     1)
            v5 = v(i + 1, j,     1)
            v6 = v(i + 1, j + 1, 1)
            v7 = v(i,     j + 1, 1)

            ncx = segment_cells(x0, x1, dx)
            ncy = segment_cells(y0, y1, dy)
            ncz = segment_cells(0.0, Lz, dz)
            blocks.append((v0, v1, v2, v3, v4, v5, v6, v7, ncx, ncy, ncz))

            bottom_face = (v0, v3, v2, v1)
            if (outlet_x_min <= xc <= outlet_x_max) and (outlet_y_min_mesh <= yc <= outlet_y_max_mesh):
                outlet_faces.append(bottom_face)
            else:
                wall_faces.append(bottom_face)

            top_face = (v4, v5, v6, v7)
            if (inlet_x_min <= xc <= inlet_x_max) and (inlet_y_min <= yc <= inlet_y_max):
                inlet_faces.append(top_face)
            else:
                wall_faces.append(top_face)

            if i == 0:
                wall_faces.append((v0, v4, v7, v3))
            if i == nxseg - 1:
                wall_faces.append((v1, v2, v6, v5))
            if j == 0:
                wall_faces.append((v0, v1, v5, v4))
            if j == nyseg - 1:
                wall_faces.append((v3, v7, v6, v2))

    vertices_txt = "\n".join(f"    ({x:.9g} {y:.9g} {z:.9g})" for x, y, z in verts)
    blocks_txt = "\n".join(
        "    hex ({} {} {} {} {} {} {} {}) ({} {} {}) simpleGrading (1 1 1)".format(*b)
        for b in blocks
    )

    def faces_txt(faces):
        return "\n".join("        ({} {} {} {})".format(*f) for f in faces)

    text = foam_header("dictionary", "blockMeshDict") + f"""
scale 1;

vertices
(
{vertices_txt}
);

blocks
(
{blocks_txt}
);

edges
(
);

boundary
(
    inlet
    {{
        type patch;
        faces
        (
{faces_txt(inlet_faces)}
        );
    }}

    outlet
    {{
        type patch;
        faces
        (
{faces_txt(outlet_faces)}
        );
    }}

    walls
    {{
        type wall;
        faces
        (
{faces_txt(wall_faces)}
        );
    }}
);

mergePatchPairs
(
);
"""

    write_text(case_dir / "system" / "blockMeshDict", text)
    print("Wrote blockMeshDict")
    print("x planes:", x_planes)
    print("y planes:", y_planes)
    print("inlet faces:", len(inlet_faces))
    print("outlet faces:", len(outlet_faces))
    print("wall faces:", len(wall_faces))


def openfoam_boundary_entries_for_U() -> str:
    cad_entry = """
    cadObstacle
    {
        type            noSlip;
    }
""" if USE_SNAPPY_CAD_OBSTACLE else ""

    return f"""
boundaryField
{{
    inlet
    {{
        type            fixedValue;
        value           uniform (0 0 {-U_in});
    }}

    outlet
    {{
        type            zeroGradient;
    }}

    walls
    {{
        type            noSlip;
    }}
{cad_entry}}}
"""


def openfoam_boundary_entries_for_p() -> str:
    cad_entry = """
    cadObstacle
    {
        type            zeroGradient;
    }
""" if USE_SNAPPY_CAD_OBSTACLE else ""

    return f"""
boundaryField
{{
    inlet
    {{
        type            zeroGradient;
    }}

    outlet
    {{
        type            fixedValue;
        value           uniform 0;
    }}

    walls
    {{
        type            zeroGradient;
    }}
{cad_entry}}}
"""


def write_field_files() -> None:
    U_text = foam_header("volVectorField", "U") + f"""
dimensions      [0 1 -1 0 0 0 0];

internalField   uniform (0 0 0);
{openfoam_boundary_entries_for_U()}
"""
    write_text(case_dir / "0" / "U", U_text)

    p_text = foam_header("volScalarField", "p") + f"""
dimensions      [0 2 -2 0 0 0 0];

internalField   uniform 0;
{openfoam_boundary_entries_for_p()}
"""
    write_text(case_dir / "0" / "p", p_text)


def write_constant_files() -> None:
    physical = foam_header("dictionary", "physicalProperties") + f"""
viscosityModel  constant;

nu              [0 2 -1 0 0 0 0] {nu};
"""
    write_text(case_dir / "constant" / "physicalProperties", physical)

    momentum = foam_header("dictionary", "momentumTransport") + """
simulationType  laminar;
"""
    write_text(case_dir / "constant" / "momentumTransport", momentum)


def write_system_files() -> None:
    control = foam_header("dictionary", "controlDict") + f"""
solver          incompressibleFluid;

startFrom       startTime;
startTime       0;
stopAt          endTime;
endTime         {of_end_time};
deltaT          1;

writeControl    timeStep;
writeInterval   {of_write_interval};
purgeWrite      0;

writeFormat     ascii;
writePrecision  8;
writeCompression off;
timeFormat      general;
timePrecision   6;

runTimeModifiable true;
"""
    write_text(case_dir / "system" / "controlDict", control)

    fv_schemes = foam_header("dictionary", "fvSchemes") + """
ddtSchemes
{
    default         steadyState;
}

gradSchemes
{
    default         Gauss linear;
}

divSchemes
{
    default                         none;
    div(phi,U)                      bounded Gauss upwind;
    div((nuEff*dev2(T(grad(U)))))   Gauss linear;
}

laplacianSchemes
{
    default         Gauss linear corrected;
}

interpolationSchemes
{
    default         linear;
}

snGradSchemes
{
    default         corrected;
}

wallDist
{
    method          meshWave;
}
"""
    write_text(case_dir / "system" / "fvSchemes", fv_schemes)

    fv_solution = foam_header("dictionary", "fvSolution") + """
solvers
{
    p
    {
        solver          GAMG;
        tolerance       1e-7;
        relTol          0.05;
        smoother        GaussSeidel;
    }

    U
    {
        solver          smoothSolver;
        smoother        symGaussSeidel;
        tolerance       1e-8;
        relTol          0.1;
    }
}

SIMPLE
{
    nNonOrthogonalCorrectors 0;

    residualControl
    {
        p               1e-4;
        U               1e-5;
    }
}

relaxationFactors
{
    fields
    {
        p               0.3;
    }
    equations
    {
        U               0.7;
    }
}
"""
    write_text(case_dir / "system" / "fvSolution", fv_solution)

    write_text(case_dir / "system" / "fvModels", foam_header("dictionary", "fvModels") + "\n")
    write_text(case_dir / "system" / "fvConstraints", foam_header("dictionary", "fvConstraints") + "\n")

    decompose = foam_header("dictionary", "decomposeParDict") + """
numberOfSubdomains 4;

method          scotch;
"""
    write_text(case_dir / "system" / "decomposeParDict", decompose)


def write_snappy_hex_mesh_dict() -> None:
    """Optional CAD obstacle meshing. Only use after blockMesh-only works."""
    text = foam_header("dictionary", "snappyHexMeshDict") + """
castellatedMesh true;
snap            true;
addLayers       false;

geometry
{
    cadObstacle.stl
    {
        type triSurfaceMesh;
        name cadObstacle;
    }
}

castellatedMeshControls
{
    maxLocalCells       1000000;
    maxGlobalCells      2000000;
    minRefinementCells  0;
    maxLoadUnbalance    0.10;
    nCellsBetweenLevels 2;

    features
    (
    );

    refinementSurfaces
    {
        cadObstacle
        {
            level (1 2);
            patchInfo
            {
                type wall;
            }
        }
    }

    resolveFeatureAngle 30;

    refinementRegions
    {
    }

    locationInMesh (0.2 0.2 0.2);

    allowFreeStandingZoneFaces true;
}

snapControls
{
    nSmoothPatch        3;
    tolerance           2.0;
    nSolveIter          30;
    nRelaxIter          5;
}

addLayersControls
{
    relativeSizes       true;
    layers
    {
    }
    expansionRatio      1.0;
    finalLayerThickness 0.3;
    minThickness        0.1;
    nGrow               0;
    featureAngle        60;
    nRelaxIter          5;
    nSmoothSurfaceNormals 1;
    nSmoothNormals      3;
    nSmoothThickness    10;
    maxFaceThicknessRatio 0.5;
    maxThicknessToMedialRatio 0.3;
    minMedianAxisAngle  90;
    nBufferCellsNoExtrude 0;
    nLayerIter          50;
}

meshQualityControls
{
    #includeEtc "caseDicts/meshQualityDict"
}

writeFlags
(
    scalarLevels
    layerSets
    layerFields
);

mergeTolerance 1e-6;
"""
    write_text(case_dir / "system" / "snappyHexMeshDict", text)


def write_allrun() -> None:
    snappy_part = ""
    if USE_SNAPPY_CAD_OBSTACLE:
        snappy_part = "snappyHexMesh -overwrite | tee log.snappyHexMesh\ncheckMesh | tee log.checkMesh.afterSnappy\n"

    allrun = f"""#!/bin/bash
cd ${{0%/*}} || exit 1
. {FOAM_BASHRC}

blockMesh | tee log.blockMesh
checkMesh | tee log.checkMesh.beforeSnappy
{snappy_part}foamRun | tee log.foamRun
rm -rf VTK
foamToVTK | tee log.foamToVTK
"""
    path = case_dir / "Allrun"
    write_text(path, allrun)
    path.chmod(0o755)


def write_openfoam_case():
    clean_case_folder()
    cad_mesh = None
    stl_path = None

    if cad_file_path.exists():
        cad_mesh, stl_path = prepare_cad_for_case()
    else:
        print("WARNING: CAD file not found. Case will still run without CAD visualization.")

    write_block_mesh_dict()
    write_field_files()
    write_constant_files()
    write_system_files()
    if USE_SNAPPY_CAD_OBSTACLE:
        write_snappy_hex_mesh_dict()
    write_allrun()

    print("OpenFOAM case written:", case_dir)
    return cad_mesh, stl_path

# ============================================================
# 4. Run OpenFOAM
# ============================================================


def foam(cmd: str, check: bool = True):
    """Run an OpenFOAM command inside the case folder."""
    full_cmd = f"source {shlex.quote(FOAM_BASHRC)} && cd {shlex.quote(str(case_dir))} && {cmd}"
    print("\n$", cmd)
    result = subprocess.run(
        ["bash", "-lc", full_cmd],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}: {cmd}")
    return result


def run_openfoam_case():
    foam("foamRun -help | head -40")
    foam("blockMesh | tee log.blockMesh")
    foam("checkMesh | tee log.checkMesh.beforeSnappy")

    if USE_SNAPPY_CAD_OBSTACLE:
        foam("snappyHexMesh -overwrite | tee log.snappyHexMesh")
        foam("checkMesh | tee log.checkMesh.afterSnappy")

    foam("foamRun | tee log.foamRun")
    foam("rm -rf VTK && foamToVTK | tee log.foamToVTK")
    print("OpenFOAM run finished.")

# ============================================================
# 5. Read foamToVTK output
# ============================================================


def numeric_time_from_path(path: Path) -> float:
    """Try to read time value from parent folder names or filename."""
    for part in reversed(path.parts):
        try:
            return float(part)
        except ValueError:
            pass
    nums = re.findall(r"(?<![A-Za-z])[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", path.stem)
    return float(nums[-1]) if nums else np.nan


def find_internal_vtk_files():
    vtk_dir = case_dir / "VTK"
    if not vtk_dir.exists():
        raise FileNotFoundError(f"VTK folder not found: {vtk_dir}. Did foamToVTK run?")

    candidates = []
    for suffix in ("*.vtu", "*.vtk", "*.vtm"):
        candidates.extend(vtk_dir.rglob(suffix))

    internal = [p for p in candidates if "boundary" not in [x.lower() for x in p.parts]]
    if not internal:
        internal = candidates

    time_files = []
    for p in internal:
        t = numeric_time_from_path(p)
        if np.isfinite(t):
            time_files.append((t, p))

    if not time_files:
        raise RuntimeError("Could not identify any VTK time files.")

    by_time = {}
    for t, p in time_files:
        if t not in by_time or p.stat().st_size > by_time[t].stat().st_size:
            by_time[t] = p

    return sorted(by_time.items(), key=lambda item: item[0])


def read_mesh(path: Path):
    mesh = pv.read(str(path))
    if isinstance(mesh, pv.MultiBlock):
        mesh = mesh.combine()
    if "U" in mesh.cell_data and "U" not in mesh.point_data:
        mesh = mesh.cell_data_to_point_data()
    return mesh


def add_velocity_magnitude(mesh):
    if "U" in mesh.point_data:
        U = np.asarray(mesh.point_data["U"])
        mesh.point_data["Umag"] = np.linalg.norm(U, axis=1)
    elif "U" in mesh.cell_data:
        U = np.asarray(mesh.cell_data["U"])
        mesh.cell_data["Umag"] = np.linalg.norm(U, axis=1)
    else:
        raise KeyError("Could not find U velocity field in VTK mesh.")
    return mesh

# ============================================================
# 6. Python/PyVista animation
# ============================================================


def make_static_geometry(plotter, cad_obstacle_visual_mesh=None):
    room_box = pv.Box(bounds=(0, Lx, 0, Ly, 0, Lz))
    plotter.add_mesh(room_box, style="wireframe", color="black", line_width=1, label="4 m room")

    inlet_disk = pv.Disc(
        center=(inlet_center_x, inlet_center_y, Lz),
        inner=0,
        outer=inlet_radius,
        normal=(0, 0, 1),
        r_res=1,
        c_res=64,
    )
    outlet_box = pv.Box(bounds=(outlet_x_min, outlet_x_max, outlet_y_min_mesh, outlet_y_max_mesh, 0, 0.02))

    plotter.add_mesh(inlet_disk, color="green", opacity=0.7, label="CFD7 inlet position")
    plotter.add_mesh(outlet_box, color="blue", opacity=0.7, label="meshed outlet strip")

    if cad_obstacle_visual_mesh is not None:
        plotter.add_mesh(cad_obstacle_visual_mesh, color="silver", opacity=0.30, label="CAD visual mesh")

    return inlet_disk, outlet_box


def save_velocity_gif(vtk_time_files, cad_obstacle_visual_mesh=None):
    velocity_gif_path = save_folder / "trackA_openfoam_velocity_slice.gif"

    final_time, final_vtk = vtk_time_files[-1]
    final_mesh = add_velocity_magnitude(read_mesh(final_vtk))

    max_animation_frames = 80
    if len(vtk_time_files) > max_animation_frames:
        indices = np.linspace(0, len(vtk_time_files) - 1, max_animation_frames).astype(int)
        anim_files = [vtk_time_files[i] for i in indices]
    else:
        anim_files = vtk_time_files

    umag_data = final_mesh.point_data.get("Umag", final_mesh.cell_data.get("Umag"))
    umag_min = 0.0
    umag_max = float(np.nanpercentile(umag_data, 99)) if len(umag_data) else U_in
    umag_max = max(umag_max, U_in)

    plotter = pv.Plotter(off_screen=True, window_size=(1100, 850))
    plotter.open_gif(str(velocity_gif_path), fps=10)
    make_static_geometry(plotter, cad_obstacle_visual_mesh)
    plotter.add_axes()
    plotter.add_legend()
    plotter.camera_position = "iso"
    plotter.camera.zoom(1.15)

    for frame_id, (t, pth) in enumerate(tqdm(anim_files, desc="Saving OpenFOAM velocity GIF", unit="frame")):
        mesh = add_velocity_magnitude(read_mesh(pth))

        try:
            slice_mesh = mesh.slice(normal="y", origin=(0.0, Ly / 2.0, 0.0))
        except Exception:
            slice_mesh = mesh

        if "velocity_slice" in plotter.actors:
            plotter.remove_actor("velocity_slice")

        plotter.add_mesh(
            slice_mesh,
            name="velocity_slice",
            scalars="Umag",
            cmap="viridis",
            clim=(umag_min, umag_max),
            opacity=0.90,
            show_scalar_bar=True,
            scalar_bar_args={"title": "|U| [m/s]"},
        )

        plotter.add_text(
            f"Track A OpenFOAM airflow | pseudo-time {t:g} | frame {frame_id+1}/{len(anim_files)}",
            name="frame_text",
            position="upper_left",
            font_size=10,
        )

        plotter.write_frame()

    plotter.close()
    print("Saved:", velocity_gif_path)
    return final_mesh, velocity_gif_path

# ============================================================
# 7. Python Lagrangian particle tracing using final OpenFOAM U
# ============================================================


def compute_particle_response_time(dp):
    return np.maximum(particle_density * dp**2 / (18.0 * air_dynamic_viscosity), 1.0e-12)


def ensure_point_velocity_mesh(mesh):
    if "U" in mesh.point_data:
        return mesh
    if "U" in mesh.cell_data:
        return mesh.cell_data_to_point_data()
    raise KeyError("No U field found for particle tracing.")


def random_points_on_inlet(n, rng):
    theta = rng.uniform(0.0, 2.0 * np.pi, n)
    r = inlet_radius * np.sqrt(rng.uniform(0.0, 1.0, n))
    x0 = inlet_center_x + r * np.cos(theta)
    y0 = inlet_center_y + r * np.sin(theta)
    z0 = np.full(n, particle_release_z)
    return np.column_stack((x0, y0, z0))


def cad_deposition_mask(points, cad_obstacle_visual_mesh, threshold=0.035):
    if cad_obstacle_visual_mesh is None or len(points) == 0:
        return np.zeros(len(points), dtype=bool)
    try:
        cloud = pv.PolyData(points)
        dist_data = cloud.compute_implicit_distance(cad_obstacle_visual_mesh)
        dist = np.asarray(dist_data.point_data["implicit_distance"])
        return np.abs(dist) < threshold
    except Exception:
        return np.zeros(len(points), dtype=bool)


def trace_particles_from_openfoam(final_mesh, cad_obstacle_visual_mesh=None, n_steps=300, particle_dt=0.02, seed=1):
    rng = np.random.default_rng(seed)
    velocity_mesh = ensure_point_velocity_mesh(final_mesh)

    def sample_U_at_points(points):
        if len(points) == 0:
            return np.empty((0, 3))

        cloud = pv.PolyData(points)
        sampled = cloud.sample(velocity_mesh, tolerance=1.0e-6)

        if "U" not in sampled.point_data:
            return np.zeros((len(points), 3))

        U = np.asarray(sampled.point_data["U"])
        valid = np.asarray(sampled.point_data.get("vtkValidPointMask", np.ones(len(points), dtype=np.uint8))).astype(bool)
        U[~valid] = 0.0
        U[~np.isfinite(U)] = 0.0
        return U

    pos = np.zeros((max_particles, 3))
    vel = np.zeros((max_particles, 3))
    status = np.zeros(max_particles, dtype=int)  # 0 unreleased, 1 active, 2 escaped, 3 wall, 4 CAD
    diameters = rng.choice(particle_diameter_options, size=max_particles, p=particle_diameter_probabilities)
    tau = compute_particle_response_time(diameters)

    next_particle = 0
    stored_pos = []
    stored_status = []

    for step in tqdm(range(n_steps), desc="Tracing particles", unit="step"):
        if step % release_every == 0 and next_particle < max_particles:
            n_release = min(particles_per_release, max_particles - next_particle)
            new_pos = random_points_on_inlet(n_release, rng)
            ids = np.arange(next_particle, next_particle + n_release)
            pos[ids] = new_pos
            vel[ids] = np.array([0.0, 0.0, -0.25])
            status[ids] = 1
            next_particle += n_release

        active = status == 1
        if np.any(active):
            active_ids = np.where(active)[0]
            U_air = sample_U_at_points(pos[active_ids])

            acc = (U_air - vel[active_ids]) / tau[active_ids, None] + g_particle[None, :]
            vel[active_ids] += particle_dt * acc
            pos[active_ids] += particle_dt * vel[active_ids]

            hit_cad = cad_deposition_mask(pos[active_ids], cad_obstacle_visual_mesh)
            if np.any(hit_cad):
                status[active_ids[hit_cad]] = 4

            active_ids = np.where(status == 1)[0]
            p_now = pos[active_ids]
            outside = (
                (p_now[:, 0] < 0.0) | (p_now[:, 0] > Lx) |
                (p_now[:, 1] < 0.0) | (p_now[:, 1] > Ly) |
                (p_now[:, 2] < 0.0) | (p_now[:, 2] > Lz)
            )

            hit_floor = p_now[:, 2] <= 0.0
            through_outlet = (
                hit_floor &
                (p_now[:, 0] >= outlet_x_min) & (p_now[:, 0] <= outlet_x_max) &
                (p_now[:, 1] >= outlet_y_min_mesh) & (p_now[:, 1] <= outlet_y_max_mesh)
            )

            status[active_ids[through_outlet]] = 2
            wall_hit = outside & (~through_outlet)
            status[active_ids[wall_hit]] = 3

            pos[:, 0] = np.clip(pos[:, 0], 0.0, Lx)
            pos[:, 1] = np.clip(pos[:, 1], 0.0, Ly)
            pos[:, 2] = np.clip(pos[:, 2], 0.0, Lz)

        stored_pos.append(pos.copy())
        stored_status.append(status.copy())

        if next_particle >= max_particles and not np.any(status == 1):
            break

    return stored_pos, stored_status


def save_particle_gif(stored_particle_pos, stored_particle_status, cad_obstacle_visual_mesh=None):
    particle_gif_path = save_folder / "trackA_particles_from_openfoam.gif"

    plotter = pv.Plotter(off_screen=True, window_size=(1100, 850))
    plotter.open_gif(str(particle_gif_path), fps=12)

    make_static_geometry(plotter, cad_obstacle_visual_mesh)
    plotter.add_axes()
    plotter.add_legend()
    plotter.camera_position = "iso"
    plotter.camera.zoom(1.15)

    def replace_points(name, pts, color, point_size):
        if name in plotter.actors:
            plotter.remove_actor(name)
        if len(pts) > 0:
            plotter.add_points(
                pts,
                name=name,
                color=color,
                point_size=point_size,
                render_points_as_spheres=True,
            )

    frames = list(zip(stored_particle_pos, stored_particle_status))
    for frame_id, (pos, status) in enumerate(tqdm(frames, desc="Saving particle GIF", unit="frame")):
        active = status == 1
        escaped = status == 2
        wall_dep = status == 3
        cad_dep = status == 4

        replace_points("active_particles", pos[active], "yellow", 8)
        replace_points("escaped_particles", pos[escaped], "blue", 8)
        replace_points("wall_deposited_particles", pos[wall_dep], "red", 10)
        replace_points("cad_deposited_particles", pos[cad_dep], "purple", 10)

        plotter.add_text(
            f"Track A particles from OpenFOAM U | frame {frame_id+1}/{len(frames)}",
            name="frame_text",
            position="upper_left",
            font_size=10,
        )

        plotter.write_frame()

    plotter.close()
    print("Saved:", particle_gif_path)
    return particle_gif_path

# ============================================================
# 8. Main
# ============================================================


def main():
    print("Python platform:", platform.platform())
    print("case_dir =", case_dir)
    print("save_folder =", save_folder)
    print("cad_file_path =", cad_file_path)
    print("cad exists?", cad_file_path.exists())
    print("inlet square patch:", (inlet_x_min, inlet_x_max, inlet_y_min, inlet_y_max, "z=Lz"))
    print("physical outlet strip width:", outlet_strip_width_y)
    print("meshed outlet strip width:", outlet_y_max_mesh - outlet_y_min_mesh)

    cad_obstacle_visual_mesh, cad_stl_path = write_openfoam_case()
    print("CAD STL path:", cad_stl_path)

    run_openfoam_case()

    vtk_time_files = find_internal_vtk_files()
    print("Found VTK time files:")
    for t, pth in vtk_time_files[:10]:
        print(t, pth)
    if len(vtk_time_files) > 10:
        print("...")

    final_mesh, velocity_gif_path = save_velocity_gif(vtk_time_files, cad_obstacle_visual_mesh)

    stored_particle_pos, stored_particle_status = trace_particles_from_openfoam(
        final_mesh,
        cad_obstacle_visual_mesh=cad_obstacle_visual_mesh,
    )
    print("Particle frames:", len(stored_particle_pos))
    print("Final status counts:", {s: int(np.sum(stored_particle_status[-1] == s)) for s in range(5)})

    particle_gif_path = save_particle_gif(
        stored_particle_pos,
        stored_particle_status,
        cad_obstacle_visual_mesh=cad_obstacle_visual_mesh,
    )

    print("Done.")
    print("Velocity GIF:", velocity_gif_path)
    print("Particle GIF:", particle_gif_path)



In [ ]:
main()
